# MVP v1: 10 общих ИНН+agr_id (февраль 2026)

Цель:
- найти 10 пар `ИНН + agr_id`, присутствующих и в Excel, и в Озере;
- рассчитать core-показатели MVP v1 memory-safe SQL:
  - `ИНН`, `Наименование`, `Номер договора`,
  - `Кол-во ТСП`, `Кол-во терминалов`, `Сумма транзакций`.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

excel_inn_col = 'ИНН'
excel_name_col = 'Наименование'
excel_contract_col = 'Номер договора'
excel_agr_id_col = 'ID договора'

top_n = 10

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods_alpha.scd1_trx')
    imp.execute('invalidate metadata ods_alpha.scd1_trx_acq')
print('Impala initialized, metadata invalidated')

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    if s == '':
        return None
    return s

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def normalize_agr_id(v):
    # Для ID договора не ограничиваем длину 10/12, как для ИНН.
    return normalize_numeric_str(v)

def normalize_text(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def detect_excel_header_row(path, required_cols, scan_rows=30):
    raw = pd.read_excel(path, header=None)
    req = {str(c).strip() for c in required_cols}
    for i in range(min(scan_rows, len(raw))):
        vals = {str(x).strip() for x in raw.iloc[i].tolist()}
        if len(req & vals) >= 2:
            return i
    return 0

## 1) Excel keys: `inn + agr_id`

In [ ]:
header_row = detect_excel_header_row(excel_path, [excel_inn_col, excel_agr_id_col, excel_contract_col, excel_name_col])
excel_raw = pd.read_excel(excel_path, header=header_row)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

for c in [excel_inn_col, excel_agr_id_col, excel_contract_col, excel_name_col]:
    if c not in excel_raw.columns:
        raise ValueError(f'Column not found in Excel: {c}')

excel_df = excel_raw[[excel_inn_col, excel_agr_id_col, excel_name_col, excel_contract_col]].copy()
excel_df['inn_norm'] = excel_df[excel_inn_col].apply(normalize_inn)
excel_df['agr_id_norm'] = excel_df[excel_agr_id_col].apply(normalize_agr_id)
excel_df['excel_name_norm'] = excel_df[excel_name_col].apply(normalize_text)
excel_df['excel_contract_norm'] = excel_df[excel_contract_col].apply(normalize_text)

excel_keys = (
    excel_df
    .dropna(subset=['inn_norm', 'agr_id_norm'])
    .drop_duplicates(['inn_norm', 'agr_id_norm'])
    .copy()
)

print('Excel rows:', len(excel_df))
print('Excel keys (inn+agr_id):', len(excel_keys))
excel_keys.head(10)

## 1.1) Диагностика отсева строк Excel при формировании `inn + agr_id`

Ниже классифицируются причины, почему строки не попадают в `excel_keys`, и выводятся 5 примеров на каждую причину.

In [ ]:
excel_diag = excel_df[[excel_inn_col, excel_agr_id_col, excel_name_col, excel_contract_col, 'inn_norm', 'agr_id_norm']].copy()
excel_diag['row_id'] = excel_diag.index

# Количество строк на ключ и число повторов (повторы после нормализации)
key_cnt = (
    excel_diag
    .dropna(subset=['inn_norm', 'agr_id_norm'])
    .groupby(['inn_norm', 'agr_id_norm'], as_index=False)
    .agg(key_rows=('row_id', 'count'))
)
excel_diag = excel_diag.merge(key_cnt, on=['inn_norm', 'agr_id_norm'], how='left')
excel_diag['key_rows'] = excel_diag['key_rows'].fillna(0).astype(int)
excel_diag['dup_rows_over_first'] = excel_diag['key_rows'].clip(lower=1) - 1

# Классификация причин
excel_diag['drop_reason'] = 'kept_in_excel_keys'
excel_diag.loc[excel_diag['inn_norm'].isna() & excel_diag['agr_id_norm'].isna(), 'drop_reason'] = 'both_inn_and_agr_id_invalid_or_empty'
excel_diag.loc[excel_diag['inn_norm'].isna() & excel_diag['agr_id_norm'].notna(), 'drop_reason'] = 'inn_invalid_or_empty'
excel_diag.loc[excel_diag['inn_norm'].notna() & excel_diag['agr_id_norm'].isna(), 'drop_reason'] = 'agr_id_invalid_or_empty_after_numeric_cleaning'
excel_diag.loc[
    excel_diag['inn_norm'].notna() & excel_diag['agr_id_norm'].notna() & (excel_diag['dup_rows_over_first'] > 0),
    'drop_reason'
] = 'duplicate_inn_agr_after_normalization'

reason_stats = (
    excel_diag
    .groupby('drop_reason', as_index=False)
    .agg(rows_cnt=('row_id', 'count'))
    .sort_values('rows_cnt', ascending=False)
    .reset_index(drop=True)
)

print('Rows total:', len(excel_diag))
print('Rows kept in excel_keys:', int((excel_diag['drop_reason'] == 'kept_in_excel_keys').sum()))
print('Rows filtered out:', int((excel_diag['drop_reason'] != 'kept_in_excel_keys').sum()))
display(reason_stats)

In [ ]:
# 5 примеров по каждой причине
sample_cols = [
    'drop_reason', 'row_id',
    excel_inn_col, 'inn_norm',
    excel_agr_id_col, 'agr_id_norm',
    excel_name_col, excel_contract_col,
    'key_rows', 'dup_rows_over_first'
]

examples_by_reason = (
    excel_diag[sample_cols]
    .sort_values(['drop_reason', 'row_id'])
    .groupby('drop_reason', group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

display(examples_by_reason)

# Отдельно показываем только причины отсева (без kept)
filtered_examples = examples_by_reason[examples_by_reason['drop_reason'] != 'kept_in_excel_keys'].copy()
display(filtered_examples)

## 2) Lake keys (SA active on month start): `inn + abs_agr_id`

In [ ]:
sql_lake_keys = f"""
with sa_active as (
    select
        regexp_replace(trim(c.c_inn), '[^0-9]', '') as inn_raw,
        cast(a.abs_agr_id as string) as agr_id_norm,
        a.n_agr,
        c.c_cmp_name as lake_name,
        a.c_agr_number as lake_contract_number
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
      and a.abs_agr_id is not null
), cleaned as (
    select
        case
          when length(inn_raw) = 9 then lpad(inn_raw, 10, '0')
          when length(inn_raw) = 11 then lpad(inn_raw, 12, '0')
          else inn_raw
        end as inn_norm,
        agr_id_norm,
        n_agr,
        lake_name,
        lake_contract_number
    from sa_active
)
select distinct inn_norm, agr_id_norm, n_agr, lake_name, lake_contract_number
from cleaned
where inn_norm is not null and inn_norm <> ''
"""

with imp:
    lake_keys = imp.fetch(sql_lake_keys)

lake_keys['inn_norm'] = lake_keys['inn_norm'].apply(normalize_inn)
lake_keys['agr_id_norm'] = lake_keys['agr_id_norm'].apply(normalize_agr_id)
lake_keys = lake_keys.dropna(subset=['inn_norm', 'agr_id_norm']).drop_duplicates(['inn_norm', 'agr_id_norm'])

print('Lake keys (inn+agr_id):', len(lake_keys))
lake_keys.head(10)

## 3) Пересечение и выбор `top10_common_keys`

In [ ]:
top10_common_keys = (
    excel_keys[['inn_norm', 'agr_id_norm']]
    .drop_duplicates()
    .merge(
        lake_keys[['inn_norm', 'agr_id_norm']].drop_duplicates(),
        on=['inn_norm', 'agr_id_norm'],
        how='inner'
    )
    .sort_values(['inn_norm', 'agr_id_norm'])
    .head(top_n)
    .reset_index(drop=True)
)

print('Intersection keys:', len(top10_common_keys))
top10_common_keys

## 4) SQL в стиле техлида (memory-safe) для core-метрик по 10 ключам

In [ ]:
if top10_common_keys.empty:
    raise ValueError('top10_common_keys is empty; cannot build SQL sample_keys')

sample_rows_sql = '\n    union all\n    '.join([
    f"select '{r.inn_norm}' as inn_norm, '{r.agr_id_norm}' as agr_id_norm"
    for r in top10_common_keys.itertuples(index=False)
])

sql_core = f"""
with sample_keys as (
    {sample_rows_sql}
),
base as (
    select
        sk.inn_norm,
        sk.agr_id_norm,
        a.n_agr,
        a.n_cmp_client,
        c.c_cmp_name as client_name,
        a.c_agr_number as contract_number
    from sample_keys sk
    join ods_alpha.scd1_agreements a
      on cast(a.abs_agr_id as string) = sk.agr_id_norm
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where a.acq_class = 'SA'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
),
retl_by_key as (
    select
        b.inn_norm,
        b.agr_id_norm,
        count(distinct m.c_nmrc) as retl_cnt
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and coalesce(m.ods_deleted_flg, '0') <> '1'
     and m.c_nmrc is not null
    group by b.inn_norm, b.agr_id_norm
),
term_by_key as (
    select
        b.inn_norm,
        b.agr_id_norm,
        count(distinct t.c_nter) as term_cnt
    from base b
    left join ods_alpha.scd1_merchants m
      on m.n_cmp = b.n_cmp_client
     and coalesce(m.ods_deleted_flg, '0') <> '1'
     and m.c_nmrc is not null
    left join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
     and coalesce(t.ods_deleted_flg, '0') <> '1'
     and cast(t.d_ter_install as date) <= cast('{month_end}' as date)
     and (t.d_ter_close is null or cast(t.d_ter_close as date) >= cast('{month_start}' as date))
    group by b.inn_norm, b.agr_id_norm
),
trx_filtered as (
    select
        t.n_trx,
        cast(t.n_amt_src as double) as n_amt_src
    from ods_alpha.scd1_trx t
    where to_date(cast(t.d_trx_orig as timestamp)) between cast('{month_start}' as date) and cast('{month_end}' as date)
      and coalesce(t.ods_deleted_flg, '0') <> '1'
      and t.c_trx_class = 'SA'
      and t.c_trx_type = 'S01'
      and t.c_nter is not null
      and coalesce(t.cf_trx_stat, '') <> 'R'
),
trx_by_key as (
    select
        b.inn_norm,
        b.agr_id_norm,
        sum(tf.n_amt_src) as trx_sum
    from base b
    left join ods_alpha.scd1_trx_acq ta
      on ta.n_agr = b.n_agr
     and coalesce(ta.ods_deleted_flg, '0') <> '1'
    left join trx_filtered tf
      on tf.n_trx = ta.n_trx
    group by b.inn_norm, b.agr_id_norm
)
select
    b.inn_norm as inn,
    b.agr_id_norm as agr_id,
    b.client_name,
    b.contract_number,
    coalesce(r.retl_cnt, 0) as retl_cnt,
    coalesce(tm.term_cnt, 0) as term_cnt,
    coalesce(tr.trx_sum, 0) as trx_sum
from base b
left join retl_by_key r on r.inn_norm = b.inn_norm and r.agr_id_norm = b.agr_id_norm
left join term_by_key tm on tm.inn_norm = b.inn_norm and tm.agr_id_norm = b.agr_id_norm
left join trx_by_key tr on tr.inn_norm = b.inn_norm and tr.agr_id_norm = b.agr_id_norm
order by b.inn_norm, b.agr_id_norm
"""

print(sql_core)

In [ ]:
with imp:
    mvp_manual_top10 = imp.fetch(sql_core)

mvp_manual_top10

## 5) Финальная сверка `Excel vs Lake` по 10 кейсам

In [ ]:
excel_top10 = (
    excel_df[['inn_norm', 'agr_id_norm', 'excel_name_norm', 'excel_contract_norm']]
    .dropna(subset=['inn_norm', 'agr_id_norm'])
    .drop_duplicates(['inn_norm', 'agr_id_norm'])
    .merge(top10_common_keys, on=['inn_norm', 'agr_id_norm'], how='inner')
)

final_check = (
    top10_common_keys
    .merge(excel_top10, on=['inn_norm', 'agr_id_norm'], how='left')
    .merge(
        mvp_manual_top10.rename(columns={
            'inn': 'inn_norm',
            'agr_id': 'agr_id_norm',
            'client_name': 'lake_name',
            'contract_number': 'lake_contract'
        }),
        on=['inn_norm', 'agr_id_norm'],
        how='left'
    )
)

final_check['name_match'] = final_check.apply(
    lambda r: str(r['excel_name_norm']).strip().lower() == str(r['lake_name']).strip().lower()
    if pd.notna(r['excel_name_norm']) and pd.notna(r['lake_name']) else None,
    axis=1
)
final_check['contract_match'] = final_check.apply(
    lambda r: str(r['excel_contract_norm']).strip().lower() == str(r['lake_contract']).strip().lower()
    if pd.notna(r['excel_contract_norm']) and pd.notna(r['lake_contract']) else None,
    axis=1
)

final_check = final_check[
    [
        'inn_norm', 'agr_id_norm',
        'excel_name_norm', 'lake_name', 'name_match',
        'excel_contract_norm', 'lake_contract', 'contract_match',
        'retl_cnt', 'term_cnt', 'trx_sum'
    ]
].sort_values(['inn_norm', 'agr_id_norm']).reset_index(drop=True)

display(top10_common_keys)
display(mvp_manual_top10)
display(final_check)

## 6) Сравнение всех рассчитываемых полей с Excel

Блок автоматически ищет в Excel колонки метрик и сравнивает их с рассчитанными полями из Lake:
- `retl_cnt` (кол-во ТСП),
- `term_cnt` (кол-во терминалов),
- `trx_sum` (сумма транзакций).

По каждой метрике считаются значения Excel/Lake, дельта и флаг совпадения.

In [ ]:
def normalize_num(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '')
    if s == '' or s.lower() in {'nan', 'none', 'null', '-'}:
        return None
    s = s.replace(',', '.')
    try:
        return float(Decimal(s))
    except Exception:
        return pd.to_numeric(s, errors='coerce')

# Строгий (без автопоиска) канонический список колонок отчета
report_columns_canonical = [
    'Филиал', 'Номер организации', 'ID договора', 'Наименование', 'ИНН',
    'Номер договора', 'Дата регистрации договора', 'Дата закрытия договора',
    'Количество дней', 'Договоры менее 90 дней', 'ВСП договора', 'Код ВСП',
    'Тариф', 'Отношение к ГК', 'Принадлежность терминального оборудования',
    'Ко-во торговых точек', 'Кол-во терминалов', 'АУР', 'Амортизация',
    'Количеств операций', 'Сумма операций', 'Комиссия \n(% с операций)',
    'Комиссия \n(₽ в месяц)', 'Комиссия эквайринга',
    'Комиссия МПС (IRF, ₽)', 'ЧОД', 'Средний экв. %', 'Средний IRF %',
    'Фин. Рез.', 'Всего ТСП', 'Эффективные ТСП', 'Неэффективные ТСП',
    'Учитывать в расчетах'
]

selected_metric_cols = {
    'retl_cnt_excel': 'Ко-во торговых точек',
    'term_cnt_excel': 'Кол-во терминалов',
    'trx_sum_excel': 'Сумма операций',
}

missing_required_metrics = [
    col for col in selected_metric_cols.values() if col not in excel_raw.columns
]
if missing_required_metrics:
    raise ValueError(f'Missing required metric columns in Excel: {missing_required_metrics}')

missing_from_canonical = [
    col for col in report_columns_canonical if col not in excel_raw.columns
]
extra_vs_canonical = [
    col for col in excel_raw.columns if col not in report_columns_canonical
]

print('Selected Excel metric columns:')
for k, v in selected_metric_cols.items():
    print(f'  {k}: {v}')

print('Missing canonical columns:', missing_from_canonical)
print('Extra columns vs canonical:', extra_vs_canonical)

In [ ]:
# Подтягиваем Excel-метрики на уровень ключа inn+agr_id
excel_metrics = excel_raw[[excel_inn_col, excel_agr_id_col]].copy()
excel_metrics['inn_norm'] = excel_metrics[excel_inn_col].apply(normalize_inn)
excel_metrics['agr_id_norm'] = excel_metrics[excel_agr_id_col].apply(normalize_agr_id)

for target_col, src_col in selected_metric_cols.items():
    if src_col is not None and src_col in excel_raw.columns:
        excel_metrics[target_col] = excel_raw[src_col].apply(normalize_num)
    else:
        excel_metrics[target_col] = None

excel_metrics = (
    excel_metrics
    .dropna(subset=['inn_norm', 'agr_id_norm'])
    .groupby(['inn_norm', 'agr_id_norm'], as_index=False)
    .agg({
        'retl_cnt_excel': 'max',
        'term_cnt_excel': 'max',
        'trx_sum_excel': 'sum',
    })
)

excel_lake_calc_cmp = (
    mvp_manual_top10
    .rename(columns={'inn': 'inn_norm', 'agr_id': 'agr_id_norm'})
    .merge(excel_metrics, on=['inn_norm', 'agr_id_norm'], how='left')
)

# Сравнение с допуском
EPS_INT = 0.0
EPS_SUM = 0.01

excel_lake_calc_cmp['retl_cnt_diff'] = excel_lake_calc_cmp['retl_cnt_excel'] - excel_lake_calc_cmp['retl_cnt']
excel_lake_calc_cmp['term_cnt_diff'] = excel_lake_calc_cmp['term_cnt_excel'] - excel_lake_calc_cmp['term_cnt']
excel_lake_calc_cmp['trx_sum_diff'] = excel_lake_calc_cmp['trx_sum_excel'] - excel_lake_calc_cmp['trx_sum']

excel_lake_calc_cmp['retl_cnt_match'] = excel_lake_calc_cmp.apply(
    lambda r: abs(r['retl_cnt_diff']) <= EPS_INT if pd.notna(r['retl_cnt_excel']) else None,
    axis=1
)
excel_lake_calc_cmp['term_cnt_match'] = excel_lake_calc_cmp.apply(
    lambda r: abs(r['term_cnt_diff']) <= EPS_INT if pd.notna(r['term_cnt_excel']) else None,
    axis=1
)
excel_lake_calc_cmp['trx_sum_match'] = excel_lake_calc_cmp.apply(
    lambda r: abs(r['trx_sum_diff']) <= EPS_SUM if pd.notna(r['trx_sum_excel']) else None,
    axis=1
)

excel_lake_calc_cmp = excel_lake_calc_cmp[[
    'inn_norm', 'agr_id_norm', 'client_name', 'contract_number',
    'retl_cnt_excel', 'retl_cnt', 'retl_cnt_diff', 'retl_cnt_match',
    'term_cnt_excel', 'term_cnt', 'term_cnt_diff', 'term_cnt_match',
    'trx_sum_excel', 'trx_sum', 'trx_sum_diff', 'trx_sum_match'
]].sort_values(['inn_norm', 'agr_id_norm']).reset_index(drop=True)

display(excel_lake_calc_cmp)

In [ ]:
calc_cmp_summary = pd.DataFrame([
    {
        'metric': 'retl_cnt',
        'excel_col': selected_metric_cols.get('retl_cnt_excel'),
        'rows_with_excel_value': int(excel_lake_calc_cmp['retl_cnt_excel'].notna().sum()),
        'match_cnt': int((excel_lake_calc_cmp['retl_cnt_match'] == True).sum()),
        'mismatch_cnt': int((excel_lake_calc_cmp['retl_cnt_match'] == False).sum()),
    },
    {
        'metric': 'term_cnt',
        'excel_col': selected_metric_cols.get('term_cnt_excel'),
        'rows_with_excel_value': int(excel_lake_calc_cmp['term_cnt_excel'].notna().sum()),
        'match_cnt': int((excel_lake_calc_cmp['term_cnt_match'] == True).sum()),
        'mismatch_cnt': int((excel_lake_calc_cmp['term_cnt_match'] == False).sum()),
    },
    {
        'metric': 'trx_sum',
        'excel_col': selected_metric_cols.get('trx_sum_excel'),
        'rows_with_excel_value': int(excel_lake_calc_cmp['trx_sum_excel'].notna().sum()),
        'match_cnt': int((excel_lake_calc_cmp['trx_sum_match'] == True).sum()),
        'mismatch_cnt': int((excel_lake_calc_cmp['trx_sum_match'] == False).sum()),
    },
])

display(calc_cmp_summary)

mismatch_examples = excel_lake_calc_cmp[
    (excel_lake_calc_cmp['retl_cnt_match'] == False) |
    (excel_lake_calc_cmp['term_cnt_match'] == False) |
    (excel_lake_calc_cmp['trx_sum_match'] == False)
].copy()

display(mismatch_examples.head(30))